# Function Calling українською

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/drive/1sPfhHIBX-OXgGneSPWAOdIxbP2PR2-my?usp=sharing
)

[![Open in GitHub](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](
https://github.com/martasumyk/ai_practice/blob/main/01-Function_Calling/02_Function_Calling_Ukr_Lapa.ipynb
)


Використовується саме fine-tuned checkpoint для function calling:

```python
TymofiiNasobko/Lapa-function-calling
```


## 0. Встановлення залежностей



In [1]:
!pip -q install -U "transformers>=4.57.1" "peft>=0.13.0" accelerate bitsandbytes sentencepiece

In [2]:
!pip install -U bitsandbytes>=0.46.

## 1. Імпорти


In [3]:
import re
import json
from typing import Any, Dict, Optional, List

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftConfig, PeftModel


## 2. Визначаємо інструменти

Для простоти використаємо три інструменти:

- `add_numbers(a, b)` — додавання двох чисел;
- `multiply_numbers(a, b)` — множення двох чисел;
- `reverse_text(text)` — перевертання тексту.

Назви функцій залишені англійською, бо для tool calling стабільніше використовувати ASCII-назви.  
Але запити, описи й відповіді — українською.


In [4]:
def add_numbers(a: int, b: int) -> int:
    """Додає два цілі числа."""
    return a + b


def multiply_numbers(a: int, b: int) -> int:
    """Множить два цілі числа."""
    return a * b


def reverse_text(text: str) -> str:
    """Повертає текст у зворотному порядку."""
    return text[::-1]


TOOLS = {
    "add_numbers": add_numbers,
    "multiply_numbers": multiply_numbers,
    "reverse_text": reverse_text,
}


In [5]:
def execute_tool_call(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """Безпечно виконує інструмент і повертає результат у словнику."""
    if tool_name not in TOOLS:
        return {
            "ok": False,
            "error": f"Невідомий інструмент: {tool_name}",
        }

    try:
        result = TOOLS[tool_name](**arguments)
        return {
            "ok": True,
            "tool": tool_name,
            "arguments": arguments,
            "result": result,
        }
    except Exception as e:
        return {
            "ok": False,
            "tool": tool_name,
            "arguments": arguments,
            "error": str(e),
        }


## 3. Приклади запитів українською


In [6]:
ukrainian_queries = [
    ("Перевернути текст", "Переверни текст: машинне навчання"),
    ("Додавання", "Додай числа 9 і 4"),
    ("Множення", "Помнож 9 на 4"),
    ("Немає відповідного інструмента", "Яка сьогодні погода у Львові?"),
]


## 4. Завантаження Lapa function-calling model

Тут використовується **саме function-calling checkpoint**:

```python
LAPA_FC_MODEL_ID = "TymofiiNasobko/Lapa-function-calling"
```


In [7]:
LAPA_FC_MODEL_ID = "TymofiiNasobko/Lapa-function-calling"

USE_4BIT = True

compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

peft_config = PeftConfig.from_pretrained(LAPA_FC_MODEL_ID)
BASE_MODEL_ID = peft_config.base_model_name_or_path

print("Function-calling adapter:", LAPA_FC_MODEL_ID)
print("Base model:", BASE_MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(LAPA_FC_MODEL_ID, trust_remote_code=True)

base_model_kwargs = {
    "device_map": "auto" if torch.cuda.is_available() else None,
    "trust_remote_code": True,
    "attn_implementation": "eager",
}

if torch.cuda.is_available() and USE_4BIT:
    base_model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
else:
    base_model_kwargs["torch_dtype"] = compute_dtype

base_model_kwargs = {k: v for k, v in base_model_kwargs.items() if v is not None}

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **base_model_kwargs)

base_model.resize_token_embeddings(len(tokenizer))

lapa_model = PeftModel.from_pretrained(base_model, LAPA_FC_MODEL_ID)
lapa_model.eval()

print("Модель завантажено для function calling:", LAPA_FC_MODEL_ID)
print("CUDA available:", torch.cuda.is_available())


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Function-calling adapter: TymofiiNasobko/Lapa-function-calling
Base model: lapa-llm/lapa-v0.1.2-instruct


model.safetensors.index.json:   0%|          | 0.00/109k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/593M [00:00<?, ?B/s]

Модель завантажено для function calling: TymofiiNasobko/Lapa-function-calling
CUDA available: True


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.out_proj.lora_A.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.self_attn.out_proj.lora_B.default.weight', 'base_model.model.model.vision_tower.encoder.layers.0.mlp.fc1.lora_A.default.weight', 'base_model.mod

## 5. Схеми інструментів і function-calling prompt

Для Lapa function-calling краще явно показати моделі, які інструменти існують, які параметри вони приймають, і попросити повернути tool call.

Підтримуються обидва формати відповіді:

```json
{"name": "add_numbers", "arguments": {"a": 9, "b": 4}}
```

або:

```text
<tool_call>{"name": "add_numbers", "arguments": {"a": 9, "b": 4}}</tool_call>
```

`none` означає, що жоден доступний інструмент не підходить.


In [10]:
from typing import Any, Dict, List

TOOLS_SCHEMA = [
    {
        "name": "add_numbers",
        "description": "Додає два цілі числа.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "integer", "description": "Перше число."},
                "b": {"type": "integer", "description": "Друге число."},
            },
            "required": ["a", "b"],
        },
    },
    {
        "name": "multiply_numbers",
        "description": "Множить два цілі числа.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "integer", "description": "Перше число."},
                "b": {"type": "integer", "description": "Друге число."},
            },
            "required": ["a", "b"],
        },
    },
    {
        "name": "reverse_text",
        "description": "Повертає текст у зворотному порядку.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {
                    "type": "string",
                    "description": "Текст, який треба перевернути.",
                },
            },
            "required": ["text"],
        },
    },
]


def build_tools_description(tools_schema: List[Dict[str, Any]]) -> str:
    lines = []

    for tool in tools_schema:
        lines.append(f"- {tool['name']}: {tool['description']}")

        props = tool["parameters"].get("properties", {})
        for arg_name, arg_info in props.items():
            arg_type = arg_info.get("type", "any")
            arg_description = arg_info.get("description", "")
            lines.append(f"  - {arg_name}: {arg_type} — {arg_description}")

    return "\n".join(lines)


SYSTEM_PROMPT = f"""
Ти асистент для function calling.
Твоє завдання — вибрати один доступний інструмент для запиту користувача.

Доступні інструменти:
{build_tools_description(TOOLS_SCHEMA)}

Правила відповіді:
- Якщо інструмент підходить, поверни тільки один tool call.
- Формат: <tool_call>{{"name": "tool_name", "arguments": {{...}}}}</tool_call>
- Якщо жоден інструмент не підходить, поверни: <tool_call>{{"name": "none", "arguments": {{}}}}</tool_call>
- Не додавай пояснень, markdown або тексту поза tool call.
""".strip()


def build_lapa_prompt(user_query: str) -> str:
    """Створює chat prompt для Lapa function-calling adapter."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    # Якщо tokenizer має chat template, використовуємо його.
    # Якщо template не підтримує tools=..., це не проблема,
    # бо інструменти вже описані в system prompt.
    if getattr(tokenizer, "chat_template", None):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tools=TOOLS_SCHEMA,
                tokenize=False,
                add_generation_prompt=True,
            )
        except TypeError:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

    # Fallback без chat template.
    return (
        f"System:\n{SYSTEM_PROMPT}\n\n"
        f"User:\n{user_query}\n\n"
        f"Assistant:\n"
    )

## 6. Парсинг і нормалізація відповіді моделі


In [12]:
import json
import re
from typing import Any, Dict, Optional


def extract_first_json_object(text: str) -> Optional[Dict[str, Any]]:
    """Витягує перший валідний JSON-об'єкт з відповіді моделі."""

    text = text.strip()

    # Якщо модель повернула <tool_call>{...}</tool_call>, спочатку беремо JSON з нього.
    tagged = re.search(
        r"<tool_call>\s*(\{.*?\})\s*</tool_call>",
        text,
        flags=re.DOTALL,
    )
    if tagged:
        try:
            obj = json.loads(tagged.group(1))
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass

    # Найчастіший випадок: модель повертає рівно JSON.
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Якщо модель обгорнула JSON у markdown-блок.
    fenced = re.search(
        r"```(?:json)?\s*(\{.*?\})\s*```",
        text,
        flags=re.DOTALL,
    )
    if fenced:
        try:
            obj = json.loads(fenced.group(1))
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass

    # Робастний пошук першого збалансованого {...}, а не greedy regex.
    start = text.find("{")

    while start != -1:
        depth = 0
        in_string = False
        escape = False

        for i in range(start, len(text)):
            ch = text[i]

            if escape:
                escape = False
                continue

            if ch == "\\":
                escape = True
                continue

            if ch == '"':
                in_string = not in_string
                continue

            if in_string:
                continue

            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1

                if depth == 0:
                    candidate = text[start : i + 1]

                    try:
                        obj = json.loads(candidate)
                        if isinstance(obj, dict):
                            return obj
                    except Exception:
                        break

        start = text.find("{", start + 1)

    return None


def normalize_tool_decision(obj: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Приводить відповідь моделі до очікуваного формату."""

    if not isinstance(obj, dict):
        return {"name": "none", "arguments": {}, "parse_error": True}

    # Деякі function-calling формати повертають {"tool_calls": [{...}]}.
    if "tool_calls" in obj and isinstance(obj["tool_calls"], list) and obj["tool_calls"]:
        first = obj["tool_calls"][0]
        if isinstance(first, dict):
            obj = first

    # Іноді function може бути вкладеним:
    # {"function": {"name": ..., "arguments": ...}}
    if isinstance(obj.get("function"), dict):
        fn = obj["function"]
        name = fn.get("name", "none")
        arguments = fn.get("arguments", {})
    else:
        # Підтримуємо кілька можливих назв полів,
        # бо open-source модель може трохи відхилитись від формату.
        name = obj.get("name") or obj.get("tool") or obj.get("function") or "none"
        arguments = obj.get("arguments") or obj.get("args") or obj.get("parameters") or {}

    if isinstance(arguments, str):
        try:
            arguments = json.loads(arguments)
        except Exception:
            arguments = {}

    if name in {"null", "None", None, "", "no_tool"}:
        name = "none"

    if name not in set(TOOLS.keys()) | {"none"}:
        return {
            "name": "none",
            "arguments": {},
            "unknown_tool": name,
        }

    if not isinstance(arguments, dict):
        arguments = {}

    return {
        "name": name,
        "arguments": arguments,
    }

## 7. Генерація tool-call JSON за допомогою Lapa


In [13]:
def generate_lapa_tool_decision(user_query: str, max_new_tokens: int = 160) -> Dict[str, Any]:
    prompt = build_lapa_prompt(user_query)

    inputs = tokenizer(prompt, return_tensors="pt").to(lapa_model.device)

    with torch.no_grad():
        output_ids = lapa_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Відрізаємо prompt, щоб залишити тільки відповідь моделі.
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=False).strip()

    parsed = extract_first_json_object(raw_output)
    decision = normalize_tool_decision(parsed)
    decision["raw_output"] = raw_output
    decision["parsed_json"] = parsed

    return decision


## 8. Виконання інструмента і фінальна відповідь українською


In [14]:
def call_lapa_with_tools(user_query: str, show_raw: bool = True) -> str:
    decision = generate_lapa_tool_decision(user_query)

    tool_name = decision["name"]
    arguments = decision["arguments"]

    if tool_name == "none":
        message = (
            "Я не бачу серед доступних інструментів такого, що може надійно виконати цей запит. "
            "Доступні лише: додавання, множення і перевертання тексту."
        )
        if show_raw:
            message += f"\nRAW MODEL OUTPUT: {decision.get('raw_output')}"
        return message

    tool_result = execute_tool_call(tool_name, arguments)

    if not tool_result["ok"]:
        return (
            "Модель спробувала викликати інструмент, але сталася помилка.\n"
            f"Рішення моделі: {decision}\n"
            f"Помилка: {tool_result}"
        )

    message = (
        f"Модель вибрала інструмент `{tool_name}` з аргументами {arguments}.\n"
        f"Результат: {tool_result['result']}"
    )
    if show_raw:
        message += f"\nRAW MODEL OUTPUT: {decision.get('raw_output')}"

    return message


## 9. Тестуємо Lapa на українських запитах


In [15]:
for label, query in ukrainian_queries:
    print("=" * 80)
    print(f"{label}\nUSER: {query}")
    try:
        answer = call_lapa_with_tools(query)
        print(answer)
    except Exception as e:
        print("Lapa error:", repr(e))


Перевернути текст
USER: Переверни текст: машинне навчання


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Я не бачу серед доступних інструментів такого, що може надійно виконати цей запит. Доступні лише: додавання, множення і перевертання тексту.
RAW MODEL OUTPUT: <think>Добре, користувач надав розмову, де вони запитують про використання інструментів для виконання завдання. Мені потрібно зрозуміти, як правильно структурувати відповідь.

По-перше, я повинен зрозуміти, що саме запитує користувач. Вони згадали про використання інструментів, тому я повинен визначити, які інструменти доступні. Дивлячись на надані інструменти, є три: add_numbers, multiply_numbers та reverse_text.

Далі я повинен визначити, який інструмент підходить для запиту користувача. У цьому випадку користувач хоче, щоб я перевернув текст "machine learning". Дивлячись на доступні інструменти, функція reverse_text призначена для виконання саме цього завдання. Вона приймає рядок як вхідні дані і повертає його у зворотному порядку.

Отже, я повинен вибрати функцію reverse_text і надати текст як аргумент. Це означає, що я повин

In [16]:
query = "Додай числа 9 і 4"

decision = generate_lapa_tool_decision(query)
decision

{'name': 'add_numbers',
 'arguments': {'a': 9, 'b': 4},
 'raw_output': '<think>Добре, користувач хоче, щоб я допоміг їм додати числа 9 і 4. Мені потрібно визначити правильний інструмент для використання. Дивлячись на доступні інструменти, є add_numbers, яка робить саме це. Вона приймає два цілих числа як вхідні дані. Отже, я повинен викликати add_numbers з a=9 і b=4. Це повинно дати користувачу правильну суму.\n</think><tool_call>{"name": "add_numbers", "arguments": {"a": 9, "b": 4}}</tool_call><end_of_turn>',
 'parsed_json': {'name': 'add_numbers', 'arguments': {'a': 9, 'b': 4}}}